In [1]:
import os
import time
import json
from dotenv import load_dotenv
from pdf2image import convert_from_path
from google import genai

# --- 1. SETUP ENVIRONMENT & CREDENTIALS ---
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("API Key tidak ditemukan! Pastikan file .env sudah benar.")

client = genai.Client(api_key=api_key)
print("SDK Google GenAI berhasil diinisialisasi.")

# --- 2. PERSIAPAN FILE & POPPLER ---
pdf_path = "Laporan Keuangan Bank Mandiri 2025.pdf" 
# PASTIKAN PATH POPPLER DI BAWAH INI SESUAI DENGAN LOKASI EKSTRAKSI KAMU
path_ke_poppler = r"C:\poppler\Library\bin" 

print(f"Memulai konversi dokumen {pdf_path}...")
try:
    # Mengambil seluruh halaman dokumen (karena di test ini hanya 9 halaman)
    halaman_gambar = convert_from_path(
        pdf_path, 
        dpi=200,
        poppler_path=path_ke_poppler
    )
    print(f"Berhasil mengonversi total {len(halaman_gambar)} halaman menjadi format gambar.")
except Exception as e:
    print(f"Gagal membaca PDF. Pastikan nama file dan path Poppler benar.\nError: {e}")
    exit()

# --- 3. PROMPT MULTIMODAL ---
ekstraksi_prompt = """
Kamu adalah AI Data Extractor yang sangat teliti. Tugasmu adalah mengekstrak seluruh informasi dari gambar halaman dokumen ini.
Ikuti aturan ketat berikut:
1. TEKS: Ekstrak semua teks naratif persis seperti aslinya.
2. TABEL: Jika terdapat tabel data, konversikan tabel tersebut menjadi format Markdown yang rapi tanpa menghilangkan satu baris/kolom pun.
3. GRAFIK/INFOGRAFIS: Jika terdapat grafik, chart, atau alur infografis, berikan penjelasan deskriptif yang mendetail tentang apa maksud gambar tersebut, termasuk angka, persentase, dan teks di dalamnya.
4. Jangan menambahkan informasi dari luar, hanya berdasarkan apa yang ada di gambar.
"""

# --- 4. EKSEKUSI API DENGAN THROTTLING ---
dokumen_rag = []
print("\n Memulai proses ekstraksi Multimodal dengan Gemini 3.5 Flash...")

for i, halaman in enumerate(halaman_gambar):
    nomor_halaman = i + 1
    print(f"[{nomor_halaman}/{len(halaman_gambar)}] Membedah halaman {nomor_halaman}...", end=" ")
    
    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=[ekstraksi_prompt, halaman]
        )
        
        # Menyimpan teks hasil ekstraksi beserta metadata
        dokumen_rag.append({
            "metadata": {"page": nomor_halaman, "source": "Laporan Bank Mandiri 2025"},
            "content": response.text
        })
        print("Berhasil!")
        
    except Exception as e:
        print(f"Gagal! (Error: {e})")
    
    # THROTTLING: Jeda 15 detik agar tidak melanggar limit 5 RPM dari Google
    # Kita tidak perlu jeda jika ini adalah halaman terakhir
    if nomor_halaman < len(halaman_gambar):
        print(f"  Pendinginan API selama 15 detik untuk menghindari Rate Limit...")
        time.sleep(15)

# --- 5. CHECKPOINTING (SIMPAN KE JSON) ---
output_file = 'knowledge_base_mandiri.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dokumen_rag, f, ensure_ascii=False, indent=4)

print(f"\n BINGO! Seluruh hasil ekstraksi telah diamankan ke dalam file '{output_file}'.")

SDK Google GenAI berhasil diinisialisasi.
Memulai konversi dokumen Laporan Keuangan Bank Mandiri 2025.pdf...
Berhasil mengonversi total 9 halaman menjadi format gambar.

 Memulai proses ekstraksi Multimodal dengan Gemini 3.5 Flash...
[1/9] Membedah halaman 1... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[2/9] Membedah halaman 2... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[3/9] Membedah halaman 3... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[4/9] Membedah halaman 4... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[5/9] Membedah halaman 5... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[6/9] Membedah halaman 6... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[7/9] Membedah halaman 7... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[8/9] Membedah halaman 8... Berhasil!


In [2]:
# Menggunakan jalur import terbaru sesuai arsitektur LangChain modern
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Membaca file JSON kita yang sangat berharga
with open('knowledge_base_mandiri.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Mengubah data JSON menjadi objek Document milik LangChain
dokumen_langchain = []
for item in raw_data:
    doc = Document(
        page_content=item['content'],
        metadata=item['metadata'] # Menyertakan metadata halaman (Wajib untuk soal test!)
    )
    dokumen_langchain.append(doc)

print(f"Berhasil memuat {len(dokumen_langchain)} halaman dari JSON.")

# 2. CHUNKING (Pemotongan Teks) DIRESTRUKTURISASI
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, # Diperbesar 4x lipat agar tabel Markdown masuk seutuhnya!
    chunk_overlap=500,
    separators=["\n\n", "\n", " ", ""] 
)

potongan_dokumen = text_splitter.split_documents(dokumen_langchain)
print(f"✂️ Dokumen berhasil dipotong menjadi {len(potongan_dokumen)} chunks ukuran besar.")

potongan_dokumen = text_splitter.split_documents(dokumen_langchain)
print(f"Dokumen berhasil dipotong menjadi {len(potongan_dokumen)} chunks.")

# 3. LOCAL EMBEDDING (Model Gratis & Cepat dari HuggingFace)
print("Mengunduh/Memuat model Local Embedding (all-MiniLM-L6-v2)...")
model_embedding_lokal = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. MEMBANGUN VECTOR DATABASE (ChromaDB)
direktori_db = "./mandiri_chroma_db"

print("Memasukkan data ke dalam ChromaDB... (Ini butuh beberapa detik)")
vector_db = Chroma.from_documents(
    documents=potongan_dokumen,
    embedding=model_embedding_lokal,
    persist_directory=direktori_db 
)

print(f"\n SUKSES BESAR! Vector Database telah tercipta dan tersimpan di folder '{direktori_db}'.")

d:\Proyek Magang\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\satri\AppData\Local\Temp\ipykernel_7024\1263474086.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Berhasil memuat 9 halaman dari JSON.
✂️ Dokumen berhasil dipotong menjadi 17 chunks ukuran besar.
Dokumen berhasil dipotong menjadi 17 chunks.
Mengunduh/Memuat model Local Embedding (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7464.33it/s]


Memasukkan data ke dalam ChromaDB... (Ini butuh beberapa detik)

 SUKSES BESAR! Vector Database telah tercipta dan tersimpan di folder './mandiri_chroma_db'.


In [7]:
import os
from dotenv import load_dotenv
from google import genai
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Setup API Key & Gemini Client
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# 2. Setup Embedding 
model_embedding_lokal = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Load Vector Database
direktori_db = "./mandiri_chroma_db"
vector_db = Chroma(persist_directory=direktori_db, embedding_function=model_embedding_lokal)
print("✅ Database ChromaDB berhasil diakses!\n")

# 4. Fungsi Utama RAG (Retrieval + Generation)
def tanya_laporan_mandiri(pertanyaan):
    print(f"Mencari jawaban untuk: '{pertanyaan}'\n")
    
    # Menarik 15 chunk dokumen untuk memastikan tabel utuh terbawa
    hasil_pencarian = vector_db.similarity_search(pertanyaan, k=15)
    
    konteks = ""
    
    for i, doc in enumerate(hasil_pencarian):
        # Menyuntikkan nomor halaman langsung ke teks untuk dibaca AI
        halaman = doc.metadata.get('page', 'Tidak diketahui')
        konteks += f"---\n[SUMBER: HALAMAN {halaman}]\n{doc.page_content}\n"
        
    print("🧠 Menyerahkan jaring konteks ke Gemini untuk dianalisis...\n")
        
    prompt_rag = f"""
    Kamu adalah asisten AI yang cerdas dan ahli dalam menganalisis dokumen keuangan Bank Mandiri.
    Tugasmu adalah menjawab pertanyaan menggunakan HANYA informasi dari 'Konteks Dokumen' di bawah ini.
    Setiap potongan konteks diawali dengan tag [SUMBER: HALAMAN X].
    
    Aturan Wajib:
    1. Sajikan data angka nominal dan persentase dengan sangat akurat.
    2. Di akhir jawabanmu, buatlah satu baris khusus berbunyi: "📌 Sumber Halaman: [Sebutkan nomor halamannya]". 
    3. HANYA sebutkan halaman tempat kamu benar-benar menemukan angka/jawaban tersebut. Jangan sebutkan halaman yang tidak relevan.

    Konteks Dokumen:
    {konteks}

    Pertanyaan: {pertanyaan}
    Jawaban:
    """
    
    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=prompt_rag
    )
    
    print("=================== JAWABAN AI ===================")
    print(response.text)
    print("==================================================")

# --- UJI COBA TECHNICAL TEST ---
pertanyaan_1 = "Berdasarkan laporan, berapa pertumbuhan nominal dan pertumbuhan persentase dari sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan tahun 2024?"
tanya_laporan_mandiri(pertanyaan_1)

print("\n\n")

# Mari kita coba tembak pertanyaan kedua dari soal tes!
pertanyaan_2 = "Apa saja saluran pengaduan yang disediakan oleh Bank Mandiri?"
tanya_laporan_mandiri(pertanyaan_2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7083.23it/s]


✅ Database ChromaDB berhasil diakses!

Mencari jawaban untuk: 'Berdasarkan laporan, berapa pertumbuhan nominal dan pertumbuhan persentase dari sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan tahun 2024?'

🧠 Menyerahkan jaring konteks ke Gemini untuk dianalisis...

=================== JAWABAN AI ===================
Berikut adalah pertumbuhan nominal dan persentase untuk sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan dengan tahun 2024 (dalam Rp juta):

1. **Sektor Tambang (Sektor Ekonomi: Tambang)**
   * **Pertumbuhan Nominal:** Rp11.614.853 juta (atau Rp11.614.853.000.000)
   * **Pertumbuhan Persentase:** 7,98%

2. **Sektor Konstruksi**
   * **Pertumbuhan Nominal:** Rp8.264.848 juta (atau Rp8.264.848.000.000)
   * **Pertumbuhan Persentase:** 8,27%

📌 Sumber Halaman: HALAMAN 4



Mencari jawaban untuk: 'Apa saja saluran pengaduan yang disediakan oleh Bank Mandiri?'

🧠 Menyerahkan jaring konteks ke Gemini untuk dianalisis...

=================== JAWABAN AI =========